Welcome to this notebook, I will teach you how I finetuned Deepseek-R1-Distill-Qwen-7B for diet recommendation using a custom full training loop with Supervised Finetuning.

The base model can be found in HuggingFace as deepseek-ai/Deepseek-R1-Distill-Qwen-7B and in the following link: https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B .

The reason why I specifically chose this model is that it was the most comfortable one to use for this task at hand and because of the chat template, which will be explained later.

This model is VictuAI-v5 because I finetuned before 4 different versions using different models and sizes.

Since I had limited resources, since Kaggle free tier offers 9 hour sessions with 2 GPUs T4 16GB RAM each, I worked as hard as possible to make it fit into the limited RAM and time while focusing also at the same time into the model's accuracy and size, being 7 billion parameters for a model a good size since it produces less hallucinations as 1.5B models, like the original VictuAI model, whose notebook can also be checked in the notebooks folder.

To make this introduction quick, I will explain more as the notebook progresses and my experiences with this notebook and others, since this is one of the dozen notebooks I created on my journey to learn how to finetune an LLM.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

The first thing we want to do is install all the packages necessary for this notebook.

The torch package is used for Pytorch types, like torch.float16, and functions. 

I would argue that the evaluate is not necessary, since we will be using the HuggingFace Transformers library, but sometimes when working with standarized datasets like Glue SST-2 dataset for Sentiment Analysis, it is more practical to use evaluate. 

The bitsandbytes package is used for the 8 bit Adam optimizer with weight decay.

The accelerate package was mainly used for GPU distributed training, managing the batches asignation to the GPUs available, which makes the training loop run without execution errors.

In [ ]:
!pip install torch evaluate bitsandbytes accelerate

I log in to HuggingFace so that I can push the model and tokenizer to the hub.

Also, there are models like Llama and Mistral that ask for you to accept their Terms of Service when working with their models, so its necessary to log in to load them.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

We load the dataset from HuggingFace, which is in the following link if you want to look at the format: https://huggingface.co/datasets/syubraj/DietRecommendation-dataset-Qwen-2.5-0.5b .

This dataset uses the Qwen chat template, which has the following structure:
<|im_start|>system
Act as a nutrition expert. Based on the user’s age, gender, height, weight, activity level, diet preference, and calorie target, 
suggest a balanced meal plan with breakfast, lunch, snacks, and dinner.<|im_end|>
<|im_start|>user
Ages: 25
Gender: Male
Height: 180 cm
Weight: 80 kg
Activity Level: Moderately Active
Dietary Preference: Omnivore
Daily Calorie Target: 2000 kcal<|im_end|>
<|im_start|>assistant
Breakfast: Oatmeal with berries and nuts
Lunch: Grilled chicken salad with mixed greens
Snack: Greek yogurt with fruit
Dinner: Salmon with roasted vegetables<|im_end|>

This is one of the reasons why I chose Deepseek as the base model. At first, I wanted to use a 7B parameters Mistral model, but since it had a different chat template and didnt have the "<|im_start|>" and "<|im_end|>", it performed poorly in the training loop, because it didnt understand the symbols as the beginning and end of a role, and even if I added the tokens to the Mistral tokenizer and changed its chat template to Qwen chat template, it kept having big losses in training, and even after finishing the finetuning, it couldn't be loaded with HuggingFace pipeline function in another notebook. 

I also tried a 7B parameters Qwen model but it was very heavy to load, even with 2 GPUs, disk offload and quantization to 4 bits. It wasn't optimized for consumer GPUs.

My base model was built on top of Qwen, so it had its chat template, and it used Deepseek-R1 reasoning before answering to the prompt, which made it a smart for this task, since providing a diet recommendation to someone implies a degree of reasoning and thought process, taking into account the person's health metrics.

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("syubraj/DietRecommendation-dataset-Qwen-2.5-0.5b")

With a checkpoint variable I could finetune any model I wanted without changing the code for the rest of cells, which allowed me to experiment with different models and sizes.

In [ ]:
checkpoint = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

Loading a 7B parameters model in 4 bits decreased enormously the amount of RAM needed to run the model in a Kaggle notebook when using the free tier.

The code basically loads the model in 4 bits precision format, which means that each of the model's 7 billion parameters will only occupy 4 bits, and doing the math, 7 billion times 0.5 bytes / parameter is 3.5GB of RAM. Also, the computations of the model, which involves matrix multiplications, will be done with Pytorch 16 bits floats, casting the model's weight into this data type, making computations precise, while increasing the degree of quantization to the double with normalized 4 bits floats (nf4), replacing the 4 bits integers to add more precision and accuracy to the model.

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

We load the model and tokenizer from the HuggingFace, adding the bitsandbytes quantization configuration and setting the distribution of the model's memory in both GPUs RAM, so that it's balanced and doesn't produce "Out of Memory" errors in the training loop.

We will finetune our model on Causal Language Modelling since our purpose is to predict the next token, based on an instruction provided and no context available.

We could have used Question Answering since you can provide your health metrics as context and a question that asks for a personalised diet, but that would force the model into searching the answer in your context provided, instead of trying to come up with something new and useful.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=bnb_config,
    device_map="auto"
)

Since the model doesn't have a padding token, we add it to both the model and tokenizer because the token will be necessary once we prepare the batches, whose tensors the GPUs require to be the same size, because we will perform matrix calculations to obtain the output.

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token = tokenizer.pad_token_id

This function adds padding and truncates the Pytorch tensors that the tokenizer will return.

In [ ]:
def tokenize_function(dataset):
    return tokenizer(
        dataset["text"],
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

The samples of the dataset are tokenized very fast because they are tokenized in parallel, which is the reason for the "batched" parameter, allowing to process multiple samples at once in batches.

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

We initilize the data collator which will organize our dataset's samples in batches for the training loop, designed for Causal Language Modelling and Mask Language Modelling, but we are interested on the first one, which is the reason why I've put the "mlm" parameter as "False".

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
tokenized_datasets

Here we remove the "text" column, since decoder-only LLM arquitectures only work with tokens, never with raw text.

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

With the DataLoader function, we create the batches for the training and evaluation samples, adjusting a batch size of 4 to deal with limited RAM, and shuffling the data for the training samples, using the data_collator function to organize the samples in batches.

In [ ]:
from torch.utils.data import DataLoader

tokenized_datasets.set_format("torch")

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    batch_size=4,
    shuffle=True,
    collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["val"],
    batch_size=4,
    collate_fn=data_collator
)

Now, I select 10 keywords, tokenize them and selecting the first token since some words may have 2 tokens because we are using Subword tokenization, but we only want the first one.

These are the words that we want to reward our model for using them more frequently, since our task at hand is basically designing a diet that includes each of those words.

In [ ]:
keytoken_ids = []
for keyword in [
    "calorie", 
    "protein", 
    "vitamin", 
    "fat", 
    "carb",
    "fiber", 
    "sugar", 
    "diet", 
    "healthy", 
    "nutrient"
]:
    ids = tokenizer.encode(keyword, add_special_tokens=False)
    keytoken_ids.append(ids[0])

This is the loss function we use, which prioritizes keyword since we increase the loss to make sure that the samples with more keywords affect more the parameter updates.

In [ ]:
from torch.nn import CrossEntropyLoss

def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
    # adjust labels and logits so that shift_logits[0] predicts shift_labels[0]
    shift_labels = inputs[..., 1:].contiguous()
    shift_logits = logits[..., :-1, :].contiguous()

    # calculates cross entropy loss for e
    loss_fct = CrossEntropyLoss(reduce=False)
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    # gets the average loss per sample
    loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
    
    # calculates how many keywords are in the samples
    weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(
        axis=[0, 2]
    )
    weights = alpha * (1.0 + weights)

    # makes weights increase losses
    weighted_loss = (loss_per_sample * weights).mean()
    return weighted_loss

We import an Adam optimizer with weight decay, which works with 8 bit precision, occupying less memory.

The job of an optimizer is to update the parameters after Pytorch automatically calculated the gradients once we compute the loss. This type of optimizer also adjusts the personal learning rate for the weights, and after updating the weights and biases, makes sure the weights are normalized after applying weight decay. 

Learning rate is adapted to the fact that the dataset has 1.3k samples for training but the model has 7B parameters, being 1e-4 a sweet spot.

Weight decay makes sure that the model doesn't generalise, but again, since I have a "small" dataset, I can't punish that much big weights.

In [ ]:
from bitsandbytes.optim import AdamW8bit

optimizer = AdamW8bit(model.parameters(), lr=1e-4, weight_decay=0.05)

Here we define more hyperparameters, with 3 epochs and a cosine learning rate scheduler.

In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_steps_per_epoch = len(train_dataloader)
num_training_steps = num_epochs * num_steps_per_epoch

lr_scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=num_steps_per_epoch * 0.1,
    num_training_steps=num_training_steps
)

The accelerate library prepares everything for the training to be distributed among the available GPUs.

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()

model, optimizer, train_dataloader, eval_dataloader, lr_scheduler = accelerator.prepare(
    model,
    optimizer,
    train_dataloader,
    eval_dataloader,
    lr_scheduler
)

Now, this is the most important optimization on this notebook.

PEFT, Parameter Efficient FineTuning, allows me to just modify the LoRA, Low Rank Adaptation, layers, which are inserted into the attention blocks, so that I only need to train the last layers of the model instead of the whole model, also leaving the base model intact, which is the reason I could do the Supervised Finetuning in a free tier Kaggle notebook.

The LoRA configuration defines matrices of 64 rank with a scaling factor of 128, since we are working with a 7B model, attention layers as the target modules, a small dropout, which is the percentage of neurons that are deactived during each iteration of the training loop to improve generalization, a task type of Causal Language Modelling and the lm_head parameters to be saved, along with the LoRA adapter parameters of the target modules.

In [ ]:
from peft import get_peft_model, LoraConfig

peft_config = LoraConfig(
    r=64, 
    lora_alpha=128,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.1, 
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head"]
)

model = get_peft_model(model, peft_config)

Finally, we run the training loop manually, instead of using the HuggingFace Trainer library since I learn more by doing the training from scratch rather than learning a library manage everything automatically, which limits my opportunities to optimize as much as possible to make sure memory is under control during training.

I use gradient accumulation to fit bigger batches without causing OOM errors, because while I can only have a batch size of 4, with 8 steps for gradient accumulation, it would be like having batch sizes of 32.

In the training loop, I do the following:
1. Compute the predictions in form of raw logits.
2. Calculate the weighted loss.
3. Show progress every once in a while.
4. Obtain the average loss.
5. Compute the gradients with backpropagation.
6. Every 8 steps, the gradients are clipped to a maximum of 1.0, the optimizer updates the parameters, the learning rate scheduler updates the learning rate, the optimizer clears the gradients and the variable that accounts for steps increases by 1.

In [ ]:
from tqdm.notebook import tqdm

gradient_accumulation_steps = 8

model.train()
completed_steps = 0
for epoch in range(num_epochs):
    for step, batch in tqdm(
        enumerate(train_dataloader, start=1), total=num_steps_per_epoch
    ):
        logits = model(batch["input_ids"]).logits
        loss = keytoken_weighted_loss(batch["input_ids"], logits, keytoken_ids)
        
        if step % 50 == 0:
            accelerator.print(
                {
                    "steps": completed_steps,
                    "loss": loss.item(),
                }
            )
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        
        if step % gradient_accumulation_steps == 0:
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            completed_steps += 1

After training, we evaluate how good the model will do in new samples, but for that we don't want the model to update its parameters, so we use the torch.no_grad() function, which temporarily deactivates the Pytorch autograd function running in the background.

We obtain a list of losses, which we later get the mean and exponentiate, to get the perplexity, which is the metric that tells us how good our model deals with new situations.

In [ ]:
model.eval()
losses = []

for batch in eval_dataloader:
    with torch.no_grad():
        outputs = model(**batch)
    losses.append(outputs.loss.item()) 
    
perplexity = torch.exp(torch.mean(torch.tensor(losses)))

print(f"Perplexity:{perplexity}")

The last thing I did was push the model to HuggingFace, which can be visited in the following URL: https://huggingface.co/sse3ele3/VictuAI-v5 .

In [ ]:
model.push_to_hub("sse3ele3/VictuAI-v5")
tokenizer.push_to_hub("sse3ele3/VictuAI-v5")